In [5]:
import sqlite3, logging

logging.basicConfig(level=logging.INFO)
BANCO = "dados.db"

def extrair_esquema(conn):
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas = [r[0] for r in cursor.fetchall()]
    esquema_str = ""
    for t in tabelas:
        cols = [c[1] for c in conn.execute(f"PRAGMA table_info('{t}');")]
        esquema_str += f"CREATE TABLE {t} ({', '.join(cols)});\n"
    return esquema_str

conn = sqlite3.connect(BANCO)
esquema = extrair_esquema(conn)
tabelas = [row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table';")]
conn.close()

print(esquema)
print(tabelas)

CREATE TABLE categories (category_id, name, description);
CREATE TABLE customers (customer_id, name, phone, email, city);
CREATE TABLE orders (order_id, customer_id, order_date, status, total_brl, payment_method, tracking_code, estimated_delivery, notes);
CREATE TABLE order_items (order_id, quantity, product_id);
CREATE TABLE products (product_id, price_brl, name, category_id, description, stock_quantity, status, specs, created_at);
CREATE TABLE promotions (promotion_id, product_id, discount_percent, description, is_active);

['categories', 'customers', 'orders', 'order_items', 'products', 'promotions']


In [9]:
pip install ollama

  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import openai
def gerar_sql_llm(pergunta, esquema_str):
    prompt = f"Esquema:\n{esquema_str}\nPergunta: \"{pergunta}\"\nSQL:"
    # Exemplo usando ChatCompletion da OpenAI (ajuste modelo conforme credenciais)
    resp = openai.ChatCompletion.create(
        model="gpt-4", temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    sql = resp.choices[0].message.content.strip()
    return sql

In [ ]:
from ollama import chat

def gerar_sql_llm(pergunta, esquema_str):

    prompt = f"""
Você é um especialista em SQL SQLite.

Sua tarefa é transformar a pergunta do usuário em uma consulta SQL SQLite.
Sempre que possível as colunas devem trazer resultados coerentes com a pergunta, ou seja, seja proativo e traga informações relevantes para a pergunta.
Por exemplo:
    - Se a pergunta for sobre o preço de um produto, a consulta deve trazer o preço do produto e informações sobre o produto, como nome, descrição, categoria, etc.
    - Se a pergunta for sobre o preço de um instrumento musical, a consulta deve considerar os descontos também.
    - Se a pergunta for sobre um tipo de instrumento musical, a consulta deve buscar por esse tipo de instrumento musical, considerando o nome, descrição, categoria e especificações da tabela de produtos e também da de categorias.

Os dados são de uma loja de instrumentos musicais.

ESQUEMA DO BANCO:
{esquema_str}

Os tipos de status possíveis são:
- products.status: 'active', 'discontinued', 'coming_soon'
- orders.status: 'pending', 'confirmed', 'shipped', 'delivered', 'cancelled'

EXEMPLO DE PERGUNTA E CONSULTA SQL (São exemplos reais):

PRIMEIRO EXEMPLO
PERGUNTA DE EXEMPLO: "Quais são os instrumentos elétricos?"
CONSULTA SQL DE EXEMPLO:

SELECT p.* 
        FROM products p
        JOIN categories c ON p.category_id = c.category_id
        WHERE 
            c.name LIKE '%elétrico%'
            OR c.name LIKE '%electric%'
            OR c.description LIKE '%elétrico%'
            OR c.description LIKE '%electric%'
            OR p.name LIKE '%elétrico%'
            OR p.description LIKE '%elétrico%' 
            OR p.specs LIKE '%elétrico%' 
            OR p.name LIKE '%electric%' 
            OR p.description LIKE '%electric%' 
            OR p.specs LIKE '%electric%'
        ;

SEGUNDO EXEMPLO
PERGUNTA DE EXEMPLO: "Qual é o instrumento mais barato e o mais caro?"
CONSULTA SQL DE EXEMPLO:

SELECT * FROM (
    SELECT
        name AS instrumento,
        description AS descricao,
        price_brl AS preco,
        'Mais Barato' AS tipo
    FROM products
    WHERE status = 'active'
    ORDER BY price_brl ASC
    LIMIT 1
)
UNION ALL
SELECT * FROM (
    SELECT
        name AS instrumento,
        description AS descricao,
        price_brl AS preco,
        'Mais Caro' AS tipo
    FROM products
    WHERE status = 'active'
    ORDER BY price_brl DESC
    LIMIT 1
);

TERCEIRO EXEMPLO
PERGUNTA DE EXEMPLO: "qual é o instrumento com o maior desconto?"
CONSULTA SQL DE EXEMPLO:

SELECT
    p.name AS instrumento,
    p.price_brl AS preco_original,
    pr.discount_percent AS desconto_percentual,
    ROUND(p.price_brl * (1 - pr.discount_percent / 100.0), 2) AS preco_final
FROM products p
JOIN promotions pr ON p.product_id = pr.product_id
WHERE pr.is_active = 1
  AND p.status = 'active'
ORDER BY pr.discount_percent DESC
LIMIT 1;

PERGUNTA:
{pergunta}

REGRAS:
- Gere somente SQL.
- Não explique a consulta.
- Não utilize Markdown.
- Não utilize ```sql.
- Use somente tabelas e colunas presentes no esquema.
- Gere apenas consultas SELECT.
- Não faça INSERT, UPDATE, DELETE, DROP, ALTER ou CREATE.

SQL:
"""

    resposta = chat(
        model="qwen2.5-coder:7b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.3
        }
    )

    sql = resposta.message.content.strip()

    return sql

In [13]:
pip install sqlglot

   ---------------------------------------- 0.0/741.8 kB ? eta -:--:--
   ---------------------------------------- 741.8/741.8 kB 21.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:
import sqlglot

def validar_sql(sql, tabelas_permitidas):
    # Exemplo simplista: só permite SELECT e verifica tabelas
    if not sql.strip().lower().startswith("select"):
        raise ValueError("Consulta deve ser SELECT")
    parsed = sqlglot.parse_one(sql)  # Valida sintaxe, lança se inválido
    # Checa se todas as tabelas usadas estão no esquema (pode-se extrair via parsed.sql())
    for table in tabelas_permitidas:
        # crude: remove tabela se presente
        if table in parsed.sql():
            continue
    # (Implementar verificação real usando parsed.find_all())
    return True

def executar_consulta(sql):
    with sqlite3.connect(BANCO) as conn:
        cursor = conn.cursor()
        try:
            cursor.execute("BEGIN")
            cursor.execute(sql)
            dados = cursor.fetchall()
            col_names = [d[0] for d in cursor.description] if cursor.description else []
            conn.commit()
            return col_names, dados
        except Exception as e:
            conn.rollback()
            logging.error("Erro ao executar SQL: %s", e)
            raise




In [64]:
# Uso:
conn = sqlite3.connect(BANCO)
esquema = extrair_esquema(conn)
tabelas = [row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table';")]
conn.close()

#pergunta = "Quantas categorias possuem?"
#pergunta = "Quais os instrumentos de corda que estão disponíveis?"
#pergunta = "Qual é o instrumento mais barato e o mais caro?"
pergunta = "Qual a bateria mais barata disponível?"

try:
    sql_gerado = gerar_sql_llm(pergunta, esquema)
    validar_sql(sql_gerado, tabelas)
    cols, resultados = executar_consulta(sql_gerado)
    print("Consulta:", sql_gerado)
    print("Colunas:", cols)
    for row in resultados:
        print(row)
except Exception as e:
    print("Erro no fluxo:", e)

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Consulta: SELECT name AS bateria, price_brl AS preco FROM products WHERE category_id = (SELECT category_id FROM categories WHERE name LIKE '%bateria%' OR description LIKE '%bateria%') AND status = 'active' ORDER BY price_brl ASC LIMIT 1;
Colunas: ['bateria', 'preco']
('Bateria Acústica Pearl Kit 2 Studio', '13285')


In [17]:
print(esquema)

CREATE TABLE categories (category_id, name, description);
CREATE TABLE customers (customer_id, name, phone, email, city);
CREATE TABLE orders (order_id, customer_id, order_date, status, total_brl, payment_method, tracking_code, estimated_delivery, notes);
CREATE TABLE order_items (order_id, quantity, product_id);
CREATE TABLE products (product_id, price_brl, name, category_id, description, stock_quantity, status, specs, created_at);
CREATE TABLE promotions (promotion_id, product_id, discount_percent, description, is_active);

